In [59]:
import pandas as pd
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

In [60]:
test_audusd = pd.read_csv('./FX_data/AUDUSD.csv')  # Replace with your actual file name
print(test_audusd.columns)
print(test_audusd.shape)

Index(['<TICKER>', '<DTYYYYMMDD>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>',
       '<CLOSE>', '<VOL>'],
      dtype='object')
(7836556, 8)


In [61]:
test_audusd['<TIME>'] = test_audusd['<TIME>'].replace(0, '000000').astype(str)
for curr_pair in [test_audusd]:
    curr_pair['datetime'] = pd.to_datetime(curr_pair['<DTYYYYMMDD>'].astype(str) + ' ' + curr_pair['<TIME>'].astype(str),
                                        format='%Y%m%d %H%M%S')
    curr_pair = curr_pair.drop(columns=['<DTYYYYMMDD>', '<TIME>'])

In [62]:
test_audusd_clean = test_audusd.drop(columns=['<DTYYYYMMDD>', '<TIME>', '<VOL>'])
# test_audusd_clean = test_audusd_clean[(test_audusd_clean['datetime'] >= '2011-01-01') & (test_audusd_clean['datetime'] < '2012-01-01')]
#set index to datetime column
test_audusd_clean = test_audusd_clean.set_index('datetime')

In [64]:
print(test_audusd_clean[test_audusd_clean.index >= pd.Timestamp('2013-01-01 18:00:00')].head(10))

                    <TICKER>  <OPEN>  <HIGH>   <LOW>  <CLOSE>
datetime                                                     
2013-01-01 18:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 19:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 20:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 21:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 22:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 23:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 20:00:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 20:10:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 20:20:00   AUDUSD  1.0393  1.0393  1.0393   1.0393
2013-01-01 20:30:00   AUDUSD  1.0393  1.0393  1.0393   1.0393


In [43]:
dup_count = test_audusd_clean.index.duplicated().sum()
print(f"Duplicate index entries before drop: {dup_count}")
if dup_count:
    test_audusd_clean = test_audusd_clean.loc[~test_audusd_clean.index.duplicated(keep='first')]
    print(f"Duplicate index entries removed. Remaining duplicates: {test_audusd_clean.index.duplicated().sum()}")

Duplicate index entries before drop: 780373
Duplicate index entries removed. Remaining duplicates: 0


In [45]:
print(test_audusd_clean.shape)
print(test_audusd_clean.head(5))
print(test_audusd_clean.tail(5))

(7056183, 5)
                    <TICKER>  <OPEN>  <HIGH>   <LOW>  <CLOSE>
datetime                                                     
2001-01-02 23:01:00   AUDUSD  0.5617  0.5617  0.5617   0.5617
2001-01-02 23:04:00   AUDUSD  0.5616  0.5616  0.5616   0.5616
2001-01-02 23:07:00   AUDUSD  0.5615  0.5615  0.5614   0.5615
2001-01-02 23:08:00   AUDUSD  0.5616  0.5616  0.5616   0.5616
2001-01-02 23:09:00   AUDUSD  0.5617  0.5618  0.5617   0.5618
                    <TICKER>  <OPEN>  <HIGH>   <LOW>  <CLOSE>
datetime                                                     
2023-09-29 18:55:00   AUDUSD  0.6433  0.6433  0.6433   0.6433
2023-09-29 18:56:00   AUDUSD  0.6433  0.6433  0.6433   0.6433
2023-09-29 18:57:00   AUDUSD  0.6433  0.6433  0.6433   0.6433
2023-09-29 18:58:00   AUDUSD  0.6433  0.6433  0.6433   0.6433
2023-09-29 18:59:00   AUDUSD  0.6433  0.6433  0.6433   0.6433


In [46]:
full_range = pd.date_range(start='2001-01-01 00:00:00', end='2023-09-30 00:00:00', freq='min')
test_audusd_reidx = test_audusd_clean.reindex(full_range)
# df_audusd_reidx[df_audusd_reidx.index == "2010-01-03 23:00:00"]
test_audusd_reidx['flag'] = 0
test_audusd_reidx['flag'] = test_audusd_reidx.index.isin(test_audusd_clean.index).astype(int)

ohlc_cols = ['<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>']
close_col = '<CLOSE>'

prev_close = test_audusd_reidx[close_col].shift(1)
mask = (test_audusd_reidx['flag'] == 0) & prev_close.notnull() & (prev_close > 0)

# broadcast previous close into the OHLC columns for masked rows
vals = prev_close[mask].values[:, None]                       # shape (k,1)
test_audusd_reidx.loc[mask, ohlc_cols] = np.repeat(vals, len(ohlc_cols), axis=1)

In [47]:

test_audusd_clean = test_audusd.drop(columns=['<DTYYYYMMDD>', '<TIME>', '<VOL>'])
#choose data from 2010 to 2011
# test_audusd_clean = test_audusd_clean[(test_audusd_clean['datetime'] >= '2010-01-01') & (test_audusd_clean['datetime'] < '2011-01-01')]
#set index to datetime column
test_audusd_clean = test_audusd_clean.set_index('datetime')

In [48]:
na_per_col = test_audusd_reidx.isna().sum()
total_nans = na_per_col.sum()
pct_per_col = (na_per_col / len(test_audusd_reidx) * 100).round(6)

print("NaNs per column:\n", na_per_col)
print("\nPercent NaNs per column (%):\n", pct_per_col)
print(f"\nTotal NaNs: {total_nans}")

NaNs per column:
 <TICKER>    4905898
<OPEN>      4663506
<HIGH>      4663506
<LOW>       4663506
<CLOSE>     4663506
flag              0
dtype: int64

Percent NaNs per column (%):
 <TICKER>    41.012078
<OPEN>      38.985742
<HIGH>      38.985742
<LOW>       38.985742
<CLOSE>     38.985742
flag         0.000000
dtype: float64

Total NaNs: 23559922


In [49]:
# create a forward-filled copy (fills all columns including '<TICKER>')
test_audusd_ffill = test_audusd_reidx.copy().ffill()

# report NaN counts before/after
nans_before = test_audusd_reidx.isna().sum().sum()
nans_after = test_audusd_ffill.isna().sum().sum()
print(f"Total NaNs before: {nans_before}, after forward-fill: {nans_after}")

# optional: For remaining leading NaNs, drop them :
mask_keep = test_audusd_ffill['<TICKER>'].notna() | test_audusd_ffill['<CLOSE>'].notna()
if mask_keep.any():
    first_idx = test_audusd_ffill.index[mask_keep][0]
    dropped = test_audusd_ffill.index.get_loc(first_idx)
    test_audusd_ffill = test_audusd_ffill.loc[first_idx:]
else:
    dropped = len(test_audusd_ffill)
    test_audusd_ffill = test_audusd_ffill.iloc[0:0]

print(f"Dropped {dropped} leading rows (if any). Remaining rows: {len(test_audusd_ffill)}")
print("Total NaNs after drop:", test_audusd_ffill.isna().sum().sum())


Total NaNs before: 23559922, after forward-fill: 14105
Dropped 2821 leading rows (if any). Remaining rows: 11959260
Total NaNs after drop: 0


In [50]:
counts = test_audusd_ffill['flag'].value_counts().sort_index()
pct_ones = test_audusd_ffill['flag'].mean() * 100

print("flag counts:\n", counts)
print(f"Percentage of 1s: {pct_ones:.6f}%")

flag counts:
 flag
0    4903077
1    7056183
Name: count, dtype: int64
Percentage of 1s: 59.001836%


In [51]:
mask0 = test_audusd_ffill['flag'] == 0
if not mask0.any():
    print("No flag==0 rows")
else:
    m = pd.Series(test_audusd_ffill.index[mask0])
    groups = (m.diff() != pd.Timedelta(minutes=1)).cumsum()
    sizes = m.groupby(groups).size()
    print(f"Number of continuous flag==0 blocks: {len(sizes)}")
    print("Sizes (minutes) of each block:", sizes.tolist())
    print("Size distribution (value counts):")
    print(sizes.value_counts().sort_index())

Number of continuous flag==0 blocks: 242392
Sizes (minutes) of each block: [2, 2, 1, 4, 5, 1, 1, 4, 125, 1, 32, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 2, 2, 1, 3, 2, 1, 2, 1, 1, 2, 1, 2, 4, 1, 2, 2, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 3, 2, 1, 1, 5, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 4, 3, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 1, 5, 3, 2, 1, 1, 1, 1, 1, 3, 5, 1, 2, 1, 1, 3, 1, 5, 4, 2, 1, 1, 1, 3, 4, 1, 4, 7, 9, 1, 2, 1, 1, 2, 1, 5, 2, 64, 66, 30, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 3, 2, 1, 1, 4, 5, 1, 2, 4, 1, 1, 1, 1, 4, 1, 1, 1, 6, 1, 2, 1, 4, 3, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2, 6, 4, 1, 5, 2, 1, 24, 1, 8, 2, 4, 1, 2, 2, 2, 6, 2, 8, 2, 5, 1, 3, 1, 1, 1, 5, 4, 3, 8, 3, 4, 5, 1, 2, 1, 1, 3, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 59, 59, 5, 30, 1, 2, 1, 2, 1, 1, 1, 1, 1, 4, 1, 1, 2, 1, 1,

In [52]:
#drop all cols except datetime index, Ticker, OHLC, flag, ret, ret_cumsum_10, category
test_audusd_ffill = test_audusd_ffill[['<TICKER>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', 'flag']]

In [53]:
#resample to 15mins
test_audusd_15m = test_audusd_ffill.resample('15T', label='right').agg({
    '<OPEN>': 'first',
    '<HIGH>': 'max',
    '<LOW>': 'min',
    '<CLOSE>': 'last',
    'flag': 'max',
})

test_audusd_15m.head(5)
print(test_audusd_15m['flag'].value_counts())

#Calculate close-to-close log returns for 4 periods for 15min data
for period in [1, 2, 3, 4]:
    test_audusd_15m[f'ret_{period}'] = (np.log(test_audusd_15m['<CLOSE>'] / test_audusd_15m['<CLOSE>'].shift(periods=period))).shift(-period)

#drop rows with time index in 00:00:15 to 03:00:00
drop_idx_15m = test_audusd_15m.between_time('00:00:15', '03:00:00').index
test_audusd_15m = test_audusd_15m.drop(index=drop_idx_15m)
test_audusd_15m.head(20)

/var/folders/sr/p42wz5n11rzcm2dvgg2x75l00000gn/T/ipykernel_15732/2830080838.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  test_audusd_15m = test_audusd_ffill.resample('15T', label='right').agg({


flag
1    530949
0    266336
Name: count, dtype: int64


,<OPEN>,<HIGH>,<LOW>,<CLOSE>,flag,ret_1,ret_2,ret_3,ret_4
2001-01-02 23:15:00,0.5617,0.5618,0.5614,0.5617,1,0.000178,-0.000178,0.000890,0.000178
2001-01-02 23:30:00,0.5617,0.5619,0.5617,0.5618,1,-0.000356,0.000712,0.000000,0.000000
2001-01-02 23:45:00,0.5618,0.5618,0.5615,0.5616,1,0.001068,0.000356,0.000356,0.000356
2001-01-03 00:00:00,0.5616,0.5622,0.5616,0.5622,1,-0.000712,-0.000712,-0.000712,-0.000712
2001-01-03 03:15:00,0.5622,0.5650,0.5622,0.5646,1,0.000885,-0.000886,-0.001773,-0.001418
2001-01-03 03:30:00,0.5646,0.5651,0.5633,0.5651,1,-0.001771,-0.002658,-0.002303,-0.002481
2001-01-03 03:45:00,0.5632,0.5649,0.5630,0.5641,1,-0.000887,-0.000532,-0.000709,0.000177
2001-01-03 04:00:00,0.5640,0.5640,0.5631,0.5636,1,0.000355,0.000177,0.001064,0.001773
2001-01-03 04:15:00,0.5631,0.5639,0.5627,0.5638,1,-0.000177,0.000709,0.001418,0.000355
2001-01-03 04:30:00,0.5638,0.5639,0.5630,0.5637,1,0.000887,0.001595,0.000532,-0.000177


In [54]:
s = test_audusd_15m['ret_1']
q20=-3.993132e-04
q40=0.000000e+00
q60=0.000000e+00
q80=4.097101e-04
q100=2.701407e-02


masks = [
    s <= q20,                                 # <= 0.2
    (s > q20) & (s <= q40),   # (0.2, 0.4]
    (s > q40) & (s <= q60),   # (0.4, 0.6]
    (s > q60) & (s <= q80),   # (0.6, 0.8]
    (s > q80) & (s <= q100),  # (0.8, 1.0]
]

ret_names = [f"ret_1_q{int(q*100)}" for q in [0.2, 0.4, 0.6, 0.8, 1.0]]  # q20, q40, q60, q80, q100

for mask, name in zip(masks, ret_names):
    test_audusd_15m.loc[mask, name] = 1
    test_audusd_15m.loc[~mask, name] = 0

test_audusd_15m.head(10)
print(test_audusd_15m[ret_names].sum())

ret_1_q20     108756.0
ret_1_q40     368353.0
ret_1_q60          0.0
ret_1_q80     114117.0
ret_1_q100    106387.0
dtype: float64


In [55]:
test_audusd_15m['o/h'] = test_audusd_15m['<OPEN>'] / test_audusd_15m['<HIGH>']
test_audusd_15m['o/l'] = test_audusd_15m['<OPEN>'] / test_audusd_15m['<LOW>']
test_audusd_15m['o/c'] = test_audusd_15m['<OPEN>'] / test_audusd_15m['<CLOSE>']
test_audusd_15m['h/l'] = test_audusd_15m['<HIGH>'] / test_audusd_15m['<LOW>']
test_audusd_15m['h/c'] = test_audusd_15m['<HIGH>'] / test_audusd_15m['<CLOSE>']
test_audusd_15m['l/c'] = test_audusd_15m['<LOW>'] / test_audusd_15m['<CLOSE>']

In [56]:
oh_q20 = 9.981470e-01
oh_q40 = 9.994795e-01
oh_q60 = 9.998925e-01
oh_q80 = 1.000000e+00
oh_q100 = 1.000000e+00
oh = [oh_q20, oh_q40, oh_q60, oh_q80, oh_q100]
ol_q20 = 1.000000e+00
ol_q40 = 1.000106e+00
ol_q60 = 1.000533e+00
ol_q80 = 1.001972e+00
ol_q100 = 1.043489e+00
ol = [ol_q20, ol_q40, ol_q60, ol_q80, ol_q100]
oc_q20 = 9.992915e-01
oc_q40 = 1.000000e+00
oc_q60 = 1.000000e+00
oc_q80 = 1.000592e+00
oc_q100 = 1.040789e+00
oc = [oc_q20, oc_q40, oc_q60, oc_q80, oc_q100]
hl_q20 = 1.000000e+00
hl_q40 = 1.000927e+00
hl_q60 = 1.002088e+00
hl_q80 = 1.004970e+00
hl_q100 = 1.045387e+00
hl = [hl_q20, hl_q40, hl_q60, hl_q80, hl_q100]
hc_q20 = 1.000000e+00
hc_q40 = 1.000114e+00
hc_q60 = 1.000551e+00
hc_q80 = 1.001598e+00
hc_q100 = 1.042679e+00
hc = [hc_q20, hc_q40, hc_q60, hc_q80, hc_q100]
lc_q20 = 9.979705e-01
lc_q40 = 9.993288e-01
lc_q60 = 9.997957e-01
lc_q80 = 1.000000e+00
lc_q100 = 1.000000e+00
lc = [lc_q20, lc_q40, lc_q60, lc_q80, lc_q100]

for col in ['o/h', 'o/l', 'o/c', 'h/l', 'h/c', 'l/c']:
    # compute thresholds
    # q_vals = [col[0], col[1], col[2], col[3], col[4]]
    quantiles = [0.2, 0.4, 0.6, 0.8, 1.0]
    if col == 'o/h':
        q_vals = oh
    elif col == 'o/l':
        q_vals = ol
    elif col == 'o/c':
        q_vals = oc
    elif col == 'h/l':
        q_vals = hl
    elif col == 'h/c':
        q_vals = hc
    elif col == 'l/c':
        q_vals = lc
    q20, q40, q60, q80, q100 = q_vals
    print(q20, q40, q60, q80, q100)

    # define exclusive masks
    masks = [
        test_audusd_15m[col] <= q20,                                 # <= 0.2
        (test_audusd_15m[col] > q20) & (test_audusd_15m[col] <= q40),   # (0.2, 0.4]
        (test_audusd_15m[col] > q40) & (test_audusd_15m[col] <= q60),   # (0.4, 0.6]
        (test_audusd_15m[col] > q60) & (test_audusd_15m[col] <= q80),   # (0.6, 0.8]
        (test_audusd_15m[col] > q80) & (test_audusd_15m[col] <= q100),  # (0.8, 1.0]
    ]

    feat_names = [f"{col}_q{int(q*100)}" for q in quantiles]  # q20, q40, q60, q80, q100

    for mask, name in zip(masks, feat_names):
        test_audusd_15m.loc[mask, name] = 1
        test_audusd_15m.loc[~mask, name] = 0

    print(test_audusd_15m[feat_names].sum())

0.998147 0.9994795 0.9998925 1.0 1.0
o/h_q20     117731.0
o/h_q40     119822.0
o/h_q60     149442.0
o/h_q80     310629.0
o/h_q100         0.0
dtype: float64
1.0 1.000106 1.000533 1.001972 1.043489
o/l_q20     303713.0
o/l_q40       8386.0
o/l_q60     151975.0
o/l_q80     120003.0
o/l_q100    113414.0
dtype: float64
0.9992915 1.0 1.0 1.000592 1.040789
o/c_q20     116387.0
o/c_q40     358546.0
o/c_q60          0.0
o/c_q80     104230.0
o/c_q100    118365.0
dtype: float64
1.0 1.000927 1.002088 1.00497 1.045387
h/l_q20     200684.0
h/l_q40     147194.0
h/l_q60     119235.0
h/l_q80     128085.0
h/l_q100    102193.0
dtype: float64
1.0 1.000114 1.000551 1.001598 1.042679
h/c_q20     271567.0
h/c_q40      15249.0
h/c_q60     181147.0
h/c_q80     111217.0
h/c_q100    118297.0
dtype: float64
0.9979705 0.9993288 0.9997957 1.0 1.0
l/c_q20     113590.0
l/c_q40     108694.0
l/c_q60     136925.0
l/c_q80     338415.0
l/c_q100         0.0
dtype: float64


In [57]:
test_audusd_15m.head(10)

,<OPEN>,<HIGH>,<LOW>,<CLOSE>,flag,ret_1,ret_2,ret_3,ret_4,ret_1_q20,...,h/c_q20,h/c_q40,h/c_q60,h/c_q80,h/c_q100,l/c_q20,l/c_q40,l/c_q60,l/c_q80,l/c_q100
2001-01-02 23:15:00,0.5617,0.5618,0.5614,0.5617,1,0.000178,-0.000178,0.000890,0.000178,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2001-01-02 23:30:00,0.5617,0.5619,0.5617,0.5618,1,-0.000356,0.000712,0.000000,0.000000,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2001-01-02 23:45:00,0.5618,0.5618,0.5615,0.5616,1,0.001068,0.000356,0.000356,0.000356,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2001-01-03 00:00:00,0.5616,0.5622,0.5616,0.5622,1,-0.000712,-0.000712,-0.000712,-0.000712,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2001-01-03 03:15:00,0.5622,0.5650,0.5622,0.5646,1,0.000885,-0.000886,-0.001773,-0.001418,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2001-01-03 03:30:00,0.5646,0.5651,0.5633,0.5651,1,-0.001771,-0.002658,-0.002303,-0.002481,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2001-01-03 03:45:00,0.5632,0.5649,0.5630,0.5641,1,-0.000887,-0.000532,-0.000709,0.000177,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2001-01-03 04:00:00,0.5640,0.5640,0.5631,0.5636,1,0.000355,0.000177,0.001064,0.001773,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
2001-01-03 04:15:00,0.5631,0.5639,0.5627,0.5638,1,-0.000177,0.000709,0.001418,0.000355,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2001-01-03 04:30:00,0.5638,0.5639,0.5630,0.5637,1,0.000887,0.001595,0.000532,-0.000177,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [58]:
if 'test_audusd_15m' not in globals():
    raise RuntimeError("test_audusd_15m not found in workspace")

out_dir = './output'
os.makedirs(out_dir, exist_ok=True)
parq_path = os.path.join(out_dir, 'test_audusd_15m.parquet')

# try writing with default engine (pyarrow or fastparquet must be installed)
test_audusd_15m.to_parquet(parq_path, index=True)
print(f"Saved test_audusd_15m to {parq_path}")

Saved test_audusd_15m to ./output/test_audusd_15m.parquet
